# 01 · Serie temporal: suscriptores activos

Primera de las tres páginas de serie temporal del informe. El objetivo no es sólo
pronosticar, sino enseñar que la serie de "suscriptores activos" **no es una sola serie**:
el mart ofrece tres definiciones de activo y sólo una de ellas deja ver la estacionalidad
real del negocio.

Contenido:

1. Las tres definiciones de activo y por qué elegimos una.
2. Descomposición STL y contrastes de estacionariedad (`analysis/utils_timeseries.py`).
3. Altas diarias: estacionalidad semanal y anual del **flujo**, y el contexto de campañas
   y aperturas de tienda.
4. Mecánica de cohortes: curva de retención, efecto del descuento de bienvenida,
   pausas de verano como efecto de **calendario** y no de **edad**.
5. Comparación de tres modelos de forecast con backtesting walk-forward:
   naive estacional (suelo), SARIMA clásico y un modelo bottom-up de cohortes.
6. Forecast a 6 meses con intervalos, y volcado a `analysis/outputs/suscriptores.json`.

> **Nota de alcance.** `docs/report_structure.md` pide para esta página "estacionalidad semanal
> y anual". La estacionalidad semanal **no puede existir en el stock** de suscriptores activos:
> nadie cancela el domingo y se reactiva el lunes. Vive en el *flujo* de altas, que sí tiene
> grano diario (`dim_customers.first_subscription_date`), y es donde se analiza — sección 3.

In [1]:
import json
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Raíz del proyecto: se busca hacia arriba para que el notebook corra igual desde
# analysis/ que desde la raíz del repo.
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data" / "warehouse.duckdb").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "analysis"))

import utils_timeseries as ts
import utils_subscriptions as us

DB_PATH = PROJECT_ROOT / "data" / "warehouse.duckdb"
OUTPUT_PATH = PROJECT_ROOT / "analysis" / "outputs" / "suscriptores.json"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# --- Parámetros del análisis ---
FORECAST_HORIZON = 6      # meses a pronosticar
BACKTEST_FOLDS = 5        # pliegues walk-forward al horizonte completo
SEASON_LENGTH = 12        # ciclo anual en datos mensuales
ALPHA = 0.05              # nivel de los contrastes
INTERVAL_LEVEL = 0.80     # nivel de los intervalos de previsión
TREND_WINDOW = 18         # meses de tendencia para proyectar altas nuevas

# Paleta categórica validada (ver skill dataviz). Se asigna por entidad, en orden fijo.
C_BLUE, C_ORANGE, C_AQUA, C_YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
C_VIOLET, C_RED = "#4a3aa7", "#e34948"
C_GRID, C_INK, C_MUTED = "#e6e6e3", "#0b0b0b", "#52514e"

PLOT_LAYOUT = dict(
    template="plotly_white", height=420,
    margin=dict(l=60, r=30, t=60, b=50),
    font=dict(color=C_INK, size=12),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
    xaxis=dict(gridcolor=C_GRID), yaxis=dict(gridcolor=C_GRID),
    hovermode="x unified",
)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("proyecto:", PROJECT_ROOT.name, "| duckdb:", DB_PATH.exists())

proyecto: capsule-club-analytics | duckdb: True


## 1. Carga del mart y las tres definiciones de activo

`dbt_project/README.md` avisa de que `fct_subscriptions_monthly` expone tres medidas de
"activo" que **no son intercambiables**:

| Columna | Qué cuenta |
|---|---|
| `is_live` | La suscripción existe y no está cancelada, aunque no facture. |
| `is_active_eom` | Activa a cierre de mes. Incluye a quien está de pausa. |
| `is_active_net_of_pauses` | Activa **y** sin pausa: lo que factura y lo que se envía. |

La diferencia no es cosmética: las pausas estacionales masivas de verano y Navidad son una
imperfección inyectada a propósito (`docs/data_imperfections.md`) y **sólo la tercera
definición las deja ver**. Contar pausas como churn sería un error; ignorarlas, también.

In [2]:
con = duckdb.connect(str(DB_PATH), read_only=True)
facts = con.sql("select * from fct_subscriptions_monthly").df()
con.close()   # DuckDB bloquea el fichero: se cierra en cuanto se ha leído.

facts["month_start"] = pd.to_datetime(facts["month_start"])
facts["cohort_month"] = pd.to_datetime(facts["cohort_month"])
for col in ["is_live", "is_active_eom", "is_active_net_of_pauses", "is_paused"]:
    facts[col] = facts[col].astype(bool)

print(f"{len(facts):,} filas · {facts.subscription_id.nunique():,} suscripciones · "
      f"{facts.month_start.min():%Y-%m} → {facts.month_start.max():%Y-%m}")

definitions = {
    "live": "is_live",
    "active_eom": "is_active_eom",
    "active_net": "is_active_net_of_pauses",
}
series = {
    name: ts.build_series(facts.assign(v=facts[col].astype(int)), "month_start", "v", freq="MS")
    for name, col in definitions.items()
}
paused = ts.build_series(facts.assign(v=facts.is_paused.astype(int)), "month_start", "v", freq="MS")
signups = ts.build_series(
    facts[facts.months_since_start == 0].assign(v=1), "month_start", "v", freq="MS"
)

active = series["active_net"]          # la serie protagonista del resto del notebook
comparison = pd.DataFrame(series).assign(paused=paused, signups=signups)
comparison.tail(14)

52,171 filas · 5,110 suscripciones · 2023-09 → 2026-08


,live,active_eom,active_net,paused,signups
month_start,,,,,
2025-07-01,1821,1746,1624,122,146
2025-08-01,1865,1775,1527,248,119
2025-09-01,1908,1835,1667,168,133
2025-10-01,2038,1944,1880,64,203
2025-11-01,2130,2054,2042,12,186
2025-12-01,2265,2182,2102,80,211
2026-01-01,2438,2346,2260,86,256
2026-02-01,2542,2443,2404,39,196
2026-03-01,2671,2526,2509,17,228


In [3]:
fig = go.Figure()
for label, color, key in [("Live (no cancelada)", C_AQUA, "live"),
                          ("Activa a cierre de mes", C_ORANGE, "active_eom"),
                          ("Activa neta de pausas", C_BLUE, "active_net")]:
    fig.add_trace(go.Scatter(x=series[key].index, y=series[key].to_numpy(), name=label,
                             mode="lines", line=dict(color=color, width=2)))
fig.update_layout(**PLOT_LAYOUT,
                  title="Tres definiciones de suscriptor activo: la de verano no es la misma",
                  yaxis_title="Suscripciones")
fig.add_annotation(x=active.index[-1], y=float(active.iloc[-1]), text="ago-2026",
                   showarrow=True, arrowhead=0, ax=-45, ay=30, font=dict(color=C_MUTED))
fig.show()

gap = (series["active_eom"] - series["active_net"])
print("Mayores brechas entre 'activa a cierre' y 'activa neta de pausas' (= pausas):")
print(gap.sort_values(ascending=False).head(6).rename("suscripciones en pausa").to_string())

Mayores brechas entre 'activa a cierre' y 'activa neta de pausas' (= pausas):
month_start
2026-08-01    445
2025-08-01    248
2026-07-01    248
2025-09-01    168
2025-07-01    122
2026-01-01     86


Las tres curvas se separan justo en julio-agosto (y, más suave, en diciembre-enero). Agosto de
2026 es el caso extremo: **445 suscripciones en pausa**, un 15,6% de las activas a cierre. Si se
pronosticara sobre `is_live` el modelo no vería nunca ese valle, y el forecast de envíos y de
caja saldría sistemáticamente alto en verano — que es justo el mes en el que el negocio
necesita acertar el aprovisionamiento.

A partir de aquí la serie de trabajo es **`is_active_net_of_pauses`**.

## 2. Descomposición y estacionariedad

Se usan las utilidades compartidas de `analysis/utils_timeseries.py` para no repetir esta
lógica en las páginas 3 y 4.

In [4]:
decomposition = ts.stl_decompose(active, period=SEASON_LENGTH, robust=True)

print(f"Fuerza de la tendencia      F_T = {decomposition.trend_strength:.3f}")
print(f"Fuerza de la estacionalidad F_S = {decomposition.seasonal_strength[SEASON_LENGTH]:.3f}")

components = decomposition.to_frame()
fig = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                    subplot_titles=("Observado", "Tendencia", f"Estacionalidad ({SEASON_LENGTH}m)", "Residuo"))
for row, (col, color) in enumerate(
        [("observed", C_BLUE), ("trend", C_ORANGE),
         (f"seasonal_{SEASON_LENGTH}", C_AQUA), ("resid", C_MUTED)], start=1):
    fig.add_trace(go.Scatter(x=components.index, y=components[col].to_numpy(), mode="lines",
                             line=dict(color=color, width=2), showlegend=False), row=row, col=1)
fig.add_hline(y=0, line=dict(color=C_GRID, width=1), row=4, col=1)
layout = {k: v for k, v in PLOT_LAYOUT.items() if k not in ("height", "yaxis", "xaxis", "legend")}
fig.update_layout(**layout, height=760, showlegend=False,
                  title="Descomposición STL de suscriptores activos netos de pausas")
fig.update_xaxes(gridcolor=C_GRID); fig.update_yaxes(gridcolor=C_GRID)
fig.show()

Fuerza de la tendencia      F_T = 0.995
Fuerza de la estacionalidad F_S = 0.702


In [5]:
# El perfil estacional medio: cuánto suma o resta cada mes del año.
seasonal = decomposition.seasonal[SEASON_LENGTH]
profile = seasonal.groupby(seasonal.index.month).mean()
profile.index = ["ene","feb","mar","abr","may","jun","jul","ago","sep","oct","nov","dic"]

fig = go.Figure(go.Bar(
    x=profile.index, y=profile.to_numpy(),
    marker_color=[C_RED if v < 0 else C_BLUE for v in profile],
    text=[f"{v:+.0f}" for v in profile], textposition="outside",
))
fig.update_layout(**{**PLOT_LAYOUT, "hovermode": "closest"}, showlegend=False,
                  title="Perfil estacional medio (suscripciones sobre la tendencia)",
                  yaxis_title="Desviación sobre la tendencia")
fig.show()
print(profile.round(1).to_string())

ene     52.3
feb     88.0
mar    105.2
abr    111.4
may     97.6
jun     87.4
jul    -78.4
ago   -378.0
sep    -34.6
oct     18.1
nov     45.3
dic     15.0


El perfil confirma la lectura de negocio: **agosto es el suelo del año** y los meses de
primavera el techo. No es churn — es la pausa estacional volviendo a la serie.

In [6]:
# Meses atípicos. Ojo con el umbral: el residuo de la STL crece con el nivel de la serie
# (sigma de 15,9 suscripciones en 2024 frente a 28,3 en 2025), así que un umbral absoluto
# sólo marcaría meses recientes. Se mide el residuo RELATIVO a la tendencia.
resid = decomposition.resid
relative_resid = (resid / decomposition.trend).dropna()
sigma_rel = float(relative_resid.std(ddof=1))
outliers = relative_resid[relative_resid.abs() > 2 * sigma_rel]

print(f"sigma del residuo absoluto : {float(resid.std(ddof=1)):.1f} suscripciones")
print(f"sigma del residuo relativo : {sigma_rel:.2%}")
print()
print("Desviación anual del residuo absoluto (prueba de la heterocedasticidad):")
print(resid.groupby(resid.index.year).std(ddof=1).round(1).to_string())
print()
print("Meses fuera de 2 sigma (residuo relativo):")
print((outliers * 100).round(2).rename("% sobre la tendencia").to_string()
      if len(outliers) else "Ninguno.")

sigma del residuo absoluto : 63.5 suscripciones
sigma del residuo relativo : 6.77%

Desviación anual del residuo absoluto (prueba de la heterocedasticidad):
month_start
2023     0.0
2024    95.1
2025    60.9
2026     0.0

Meses fuera de 2 sigma (residuo relativo):
month_start
2024-08-01    32.16
Freq: MS


Sale un único mes atípico, **agosto de 2024, un 32% por encima de la tendencia**, y tiene una
explicación concreta que no es una campaña: fue el agosto con menos pausas de los tres. La tasa
de pausa de agosto oscila entre el 9,7% y el 15,6% según el año, y 2024 marcó el mínimo, así
que ese mes se quedó muy por encima de lo que la componente estacional esperaba para un agosto.

Es un buen recordatorio de que un residuo grande no significa "pasó algo raro en el negocio":
aquí significa que la estacionalidad media no describe bien un año concreto.

Es además el límite de lo que este mart puede decir sobre campañas: los datos de campaña
viven en `fct_marketing_attribution` y se tratan en la página 6 del informe.

In [7]:
# maxlag explícito: con 36 puntos, la regla por defecto de statsmodels (Schwert) pide hasta
# 10 retardos — casi un tercio de la muestra — y el contraste deja de ser fiable: llega a
# declarar estacionaria la serie en log, que tiene una tendencia evidente. Se acota con la
# regla corta 4*(n/100)^(2/9); dentro de ese tope el AIC sigue eligiendo el retardo.
MAXLAG = int(np.ceil(4 * (len(active) / 100) ** (2 / 9)))
print(f"n = {len(active)} meses -> maxlag = {MAXLAG} (por defecto habrían sido "
      f"{int(np.ceil(12 * (len(active) / 100) ** 0.25))})")
print()

stat_kwargs = dict(season_length=SEASON_LENGTH, alpha=ALPHA, regression="ct", maxlag=MAXLAG)
report_level = ts.stationarity_report(active, **stat_kwargs)
report_log = ts.stationarity_report(np.log(active), **stat_kwargs)

print("Serie en nivel :", report_level.summary())
print("Serie en log   :", report_log.summary())
print()
print(f"ADF  estadístico={report_level.adf.statistic:.3f}  p={report_level.adf.p_value:.4f}  "
      f"lags={report_level.adf.used_lags}  H0: {report_level.adf.null_hypothesis}")
print(f"KPSS estadístico={report_level.kpss.statistic:.3f}  p={report_level.kpss.p_value:.4f}"
      f"{' (recortado por tabla)' if report_level.kpss.p_value_is_bound else ''}  "
      f"H0: {report_level.kpss.null_hypothesis}")
print(f"\nDiferenciación sugerida: d={report_level.n_diffs}, D={report_level.n_seasonal_diffs} "
      f"(m={report_level.season_length})")

n = 36 meses -> maxlag = 4 (por defecto habrían sido 10)

Serie en nivel : discrepancia: ADF no rechaza la raíz unitaria pero KPSS no rechaza la estacionariedad | ADF p=0.0598 | KPSS p=0.1000 (recortado) | d=2, D=1 (m=12)
Serie en log   : no estacionaria (ADF y KPSS coinciden) | ADF p=0.1166 | KPSS p=0.0100 (recortado) | d=0, D=0 (m=12)

ADF  estadístico=-3.341  p=0.0598  lags=4  H0: raíz unitaria (no estacionaria)
KPSS estadístico=0.095  p=0.1000 (recortado por tabla)  H0: estacionaria

Diferenciación sugerida: d=2, D=1 (m=12)


Lectura de los contrastes: con sólo 36 puntos, **ADF no rechaza la raíz unitaria** ni siquiera
permitiendo tendencia, mientras que **KPSS tampoco rechaza la estacionariedad**. La discrepancia
no es un fallo, es lo esperable en una serie corta y fuertemente tendencial: los dos contrastes
tienen poca potencia con esta longitud.

Merece la pena insistir en el número de retardos, porque es una trampa fácil: **dejando el
tope de retardos por defecto, el ADF declara estacionaria la serie en logaritmos** (p = 0,001)
pese a que crece de 32 a 2.412 suscriptores. Con 36 observaciones la regla de Schwert pide hasta
10 retardos y el contraste se queda sin grados de libertad. Acotando el tope a 4, el mismo
contraste devuelve p = 0,12 y deja de rechazar. Es un buen recordatorio de que un p-valor no
sustituye a mirar la serie.

La conclusión operativa es que hay que diferenciar antes de modelar con SARIMA. La sugerencia
automática sale `d=2, D=1`, pero el modelo de abajo usa `d=1, D=1`: sobre 36 puntos la segunda
diferencia regular añade más varianza que señal, y el backtesting confirma que la versión con
`d=1` pronostica mejor.

Se trabaja además en **logaritmos**: el crecimiento es cercano al multiplicativo y la varianza
sube con el nivel, así que el log estabiliza la varianza y convierte la estacionalidad en
aditiva.

## 3. Altas diarias: dónde vive la estacionalidad semanal

El informe pide estacionalidad semanal y anual para esta página. La semanal **no puede estar en
la serie de activos**: un stock de suscriptores no oscila por día de la semana, porque nadie
cancela el domingo y se reactiva el lunes. Donde sí existe es en el **flujo de altas**, que es
lo que alimenta el stock.

`dim_subscriptions` da ese flujo a grano diario, así que se descompone con MSTL —dos
estacionalidades a la vez, semanal y anual— usando `mstl_decompose` de las utilidades
compartidas.

> **Sobre el grano.** Este mart se creó precisamente para esta sección. Antes la serie salía de
> `dim_customers.first_subscription_date`, que cuenta *primeras suscripciones por cliente*
> (5.047) y no *altas de suscripción* (5.110): se perdían las resuscripciones de quien ya había
> sido cliente. Ahora el evento contado es el correcto y la serie cuadra con los conteos de
> cohorte del resto del notebook.

In [8]:
con = duckdb.connect(str(DB_PATH), read_only=True)
daily_raw = con.sql("""
    select start_date as ds, count(*) as n
    from dim_subscriptions
    group by 1
""").df()
con.close()

HISTORY_END = active.index.max() + pd.offsets.MonthEnd(0)
daily_signups = ts.build_series(daily_raw, "ds", "n", freq="D",
                                start=active.index.min(), end=HISTORY_END)

signup_decomposition = ts.mstl_decompose(daily_signups, periods=ts.infer_seasonal_periods(daily_signups))
print(f"{len(daily_signups)} días · {daily_signups.sum():.0f} altas · media {daily_signups.mean():.2f}/día")
print(f"periodos modelados: {signup_decomposition.periods}")
for period, strength in signup_decomposition.seasonal_strength.items():
    etiqueta = "semanal" if period == 7 else "anual"
    print(f"  fuerza estacional {etiqueta:8s} (m={period:3d}) = {strength:.3f}")
print(f"  fuerza de la tendencia                  = {signup_decomposition.trend_strength:.3f}")

DAYS = ["lun", "mar", "mié", "jue", "vie", "sáb", "dom"]
raw_by_dow = daily_signups.groupby(daily_signups.index.dayofweek).mean()
weekly_component = signup_decomposition.seasonal[7]
comp_by_dow = weekly_component.groupby(weekly_component.index.dayofweek).mean()

weekday_table = pd.DataFrame({
    "altas_medias": raw_by_dow.to_numpy().round(2),
    "componente_mstl": comp_by_dow.to_numpy().round(3),
}, index=DAYS)
weekend_ratio = float(raw_by_dow.loc[[5, 6]].mean() / raw_by_dow.loc[[0, 1, 2, 3, 4]].mean())
print()
print(weekday_table.to_string())
print()
print(f"Fin de semana frente a laborable: {weekend_ratio:.3f} "
      f"({(1 - weekend_ratio) * 100:.1f}% menos de altas)")

1096 días · 5110 altas · media 4.66/día
periodos modelados: (7, 365)
  fuerza estacional semanal  (m=  7) = 0.180
  fuerza estacional anual    (m=365) = 0.514
  fuerza de la tendencia                  = 0.560

     altas_medias  componente_mstl
lun          5.09            0.583
mar          4.90            0.449
mié          4.66           -0.170
jue          4.96            0.187
vie          4.80            0.057
sáb          4.18           -0.646
dom          4.06           -0.446

Fin de semana frente a laborable: 0.844 (15.6% menos de altas)


In [9]:
fig = make_subplots(rows=2, cols=1, vertical_spacing=0.16,
                    subplot_titles=("Componente semanal (altas sobre la media del día)",
                                    "Componente anual (altas sobre la media del año)"))
fig.add_trace(go.Bar(x=DAYS, y=comp_by_dow.to_numpy(),
                     marker_color=[C_RED if v < 0 else C_BLUE for v in comp_by_dow],
                     text=[f"{v:+.2f}" for v in comp_by_dow], textposition="outside",
                     showlegend=False), row=1, col=1)
annual = signup_decomposition.seasonal[365]
fig.add_trace(go.Scatter(x=annual.index, y=annual.to_numpy(), mode="lines",
                         line=dict(color=C_AQUA, width=2), showlegend=False), row=2, col=1)
fig.add_hline(y=0, line=dict(color=C_GRID, width=1), row=2, col=1)
layout3 = {k: v for k, v in PLOT_LAYOUT.items() if k not in ("height", "xaxis", "yaxis", "legend", "hovermode")}
fig.update_layout(**layout3, height=640, showlegend=False, hovermode="closest",
                  title="Altas diarias: las dos estacionalidades, separadas con MSTL")
fig.update_xaxes(gridcolor=C_GRID); fig.update_yaxes(gridcolor=C_GRID)
fig.show()

Las dos estacionalidades existen y son de tamaño muy distinto. La **semanal** es nítida pero
modesta (F_S ≈ 0,18): el fin de semana se dan de alta un 15,6% menos de clientes que un día
laborable, con el lunes como pico. Es un patrón de canal, no de negocio — la gente contrata
desde el trabajo.

La **anual** pesa bastante más (F_S ≈ 0,51) y es la que importa para planificar: es el mismo
ciclo que ya se veía en el stock, con el verano hundido y el arranque de año fuerte.

Para el forecast esto tiene una consecuencia práctica: como el modelo trabaja a grano mensual,
la estacionalidad semanal **se promedia y desaparece**. No se pierde nada al ignorarla en el
modelo; sí se perdería si alguien quisiera planificar la carga operativa de un día concreto.

### Campañas y aperturas de tienda

Lo que sigue es **contexto, no atribución**. Se superpone la presión de marketing y las aperturas
de boutique sobre las altas mensuales para ver si coinciden con los tramos donde la serie se
acelera. Repartir el mérito entre canales es el trabajo de la página 6, que usa modelos de
atribución de verdad; aquí sólo se mira si hay coincidencia temporal.

In [10]:
con = duckdb.connect(str(DB_PATH), read_only=True)
marketing_monthly = con.sql("""
    select touchpoint_month as m, count(*) as touchpoints, sum(cost_eur) as spend_eur
    from fct_marketing_attribution
    group by 1
""").df()
# La apertura de una boutique se deduce de su primera venta en fct_shop_orders. Coincide al día
# con la fecha de apertura real para las cuatro que abren dentro del histórico, así que no hace
# falta exponer la tabla de tiendas como mart sólo para esto.
store_openings = con.sql("""
    select store_id, any_value(store_city) as city, min(order_date) as opened_at
    from fct_shop_orders
    where store_id is not null
    group by 1
""").df()
con.close()

marketing_touchpoints = ts.build_series(marketing_monthly, "m", "touchpoints", freq="MS")
marketing_spend = ts.build_series(marketing_monthly, "m", "spend_eur", freq="MS")
store_openings["opened_at"] = pd.to_datetime(store_openings["opened_at"])
# Margen de un mes: una boutique que ya estaba abierta cuando arranca el histórico vende desde
# el primer día, así que su "primera venta" no es una apertura. Sin el margen se colarían
# Barcelona y Sevilla, que simplemente venden el 2 y el 3 de septiembre de 2023.
first_month_end = active.index.min() + pd.Timedelta(days=30)
openings = store_openings[store_openings.opened_at > first_month_end].sort_values("opened_at")
print("Aperturas de boutique dentro del histórico:")
print(openings.to_string(index=False))

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=("Altas de suscripción por mes",
                                    "Presión de marketing: touchpoints por mes"))
fig.add_trace(go.Scatter(x=signups.index, y=signups.to_numpy(), mode="lines",
                         line=dict(color=C_BLUE, width=2), showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=marketing_touchpoints.index, y=marketing_touchpoints.to_numpy(),
                         mode="lines", line=dict(color=C_ORANGE, width=2),
                         showlegend=False), row=2, col=1)
for _, row in openings.iterrows():
    for r in (1, 2):
        fig.add_vline(x=row.opened_at, line=dict(color=C_MUTED, width=1, dash="dot"), row=r, col=1)
    fig.add_annotation(x=row.opened_at, yref="y domain", y=1.0, row=1, col=1,
                       text=row.city, showarrow=False, yanchor="bottom",
                       font=dict(color=C_MUTED, size=10))
layout4 = {k: v for k, v in PLOT_LAYOUT.items() if k not in ("height", "xaxis", "yaxis", "legend")}
fig.update_layout(**layout4, height=600, showlegend=False,
                  title="Altas frente a presión de marketing y aperturas de boutique")
fig.update_xaxes(gridcolor=C_GRID); fig.update_yaxes(gridcolor=C_GRID)
fig.show()

print()
print("Touchpoints por mes, últimos 8:")
print(marketing_touchpoints.tail(8).astype(int).to_string())

Aperturas de boutique dentro del histórico:
store_id     city  opened_at
   ST008    Palma 2024-05-20
   ST006   Málaga 2024-10-07
   ST007 Zaragoza 2024-12-31
   ST005   Bilbao 2025-08-07



Touchpoints por mes, últimos 8:
m
2026-01-01    2007
2026-02-01    1701
2026-03-01    1992
2026-04-01    1811
2026-05-01    1695
2026-06-01    1427
2026-07-01    1026
2026-08-01     496
Freq: MS


**Lectura, con cautela.** Las aperturas de boutique no dejan un salto visible en las altas de
suscripción: tiene sentido, porque una boutique capta sobre todo compra puntual, y el club se
vende por canales digitales. La coincidencia que sí se ve es entre los tramos de aceleración de
altas y los trimestres de más presión de marketing, pero eso es correlación en dos series que
además crecen las dos — no prueba nada por sí solo.

**Y un aviso que ya está resuelto, no pendiente.** Los touchpoints caen un 71% entre mayo y
agosto de 2026 mientras las altas siguen subiendo. Es tentador leerlo como "captamos más
gastando menos", y sería un titular estupendo para la página 7. **Es un artefacto de la ventana
de observación**: cada recorrido de marketing se construye hacia atrás desde la fecha de alta,
dentro de una ventana de consideración de unos 70 días, así que quien se dé de alta después del
cierre del histórico no aporta ningún touchpoint. Los últimos ~70 días están censurados por la
derecha por construcción.

La regla práctica, que las páginas 6 y 7 tendrán que respetar: **cualquier métrica de volumen o
de coste por touchpoint debe excluir los últimos 70 días del histórico**, o comparar sólo
periodos completos.

## 4. Mecánica de cohortes

Aquí es donde el análisis deja de tratar la serie como una caja negra. Un agregado de
suscriptores es la suma de cohortes de alta que envejecen, cada una con su curva de retención,
más las altas nuevas — y por encima de todo eso, un efecto de **calendario** (las pausas) que
no depende de la edad de la suscripción.

La descomposición que usa el modelo es:

```
activos_netos(t) = [ cohortes vivas · retención(edad) + altas nuevas · retención(edad) ] · (1 − tasa_pausa(mes))
                   \_________________ efecto de EDAD ______________/   \___ efecto de CALENDARIO ___/
```

Separar las dos cosas importa: si la curva de retención se estimara sobre `is_active_net_of_pauses`,
las pausas de agosto se colarían como "churn a los N meses" en cohortes que se dieron de alta
en meses distintos, y contaminarían la curva. Por eso **la retención se mide sobre
`is_active_eom`** (que ignora pausas) y la pausa se aplica después como factor de calendario.

In [11]:
# --- Curva de retención, ponderada por tamaño de cohorte ---
MIN_COHORT = 10   # cohortes con muy pocas altas dan ratios inestables

# La mecánica vive en analysis/utils_subscriptions.py, no aquí: el notebook 02 la reutiliza
# para pronosticar el canal de suscripción, que es el 40% del ingreso. Tenerla en un módulo
# compartido evita que aquel notebook dependa del JSON de salida de éste.
curve = us.build_retention_curve(facts, min_cohort=MIN_COHORT)
retention = curve.values
hazard = curve.hazard
n_cohorts_at_age = curve.n_cohorts_by_age

pivot = facts.pivot_table(index="cohort_month", columns="months_since_start",
                          values="is_active_eom", aggfunc="sum")
cohort_size = pivot[0]

curve_table = pd.DataFrame({
    "retencion": retention.round(4),
    "hazard": hazard.round(4),
    "cohortes_observadas": n_cohorts_at_age,
}).head(19)
print(curve_table.to_string())

                    retencion  hazard  cohortes_observadas
months_since_start                                        
0                      1.0000     NaN                   36
1                      0.9423  0.0577                   35
2                      0.8482  0.0998                   34
3                      0.8148  0.0394                   33
4                      0.7757  0.0479                   32
5                      0.7375  0.0493                   31
6                      0.7021  0.0480                   30
7                      0.6829  0.0273                   29
8                      0.6622  0.0304                   28
9                      0.6434  0.0283                   27
10                     0.6194  0.0373                   26
11                     0.5905  0.0468                   25
12                     0.5724  0.0306                   24
13                     0.5553  0.0299                   23
14                     0.5375  0.0320                   

In [12]:
# Retención (acumulada) y hazard (mensual) miden cosas distintas en escalas distintas, así que
# van en dos paneles y no en un doble eje: superponerlos invita a leer cruces que no existen.
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.09,
                    subplot_titles=("Retención acumulada (%)", "Churn del mes (%)"))
fig.add_trace(go.Scatter(x=retention.index, y=retention.to_numpy() * 100, name="Retención",
                         mode="lines+markers", line=dict(color=C_BLUE, width=2),
                         marker=dict(size=7), showlegend=False), row=1, col=1)
fig.add_trace(go.Bar(x=hazard.index, y=hazard.to_numpy() * 100, name="Churn del mes",
                     marker_color=C_ORANGE, showlegend=False), row=2, col=1)
layout2 = {k: v for k, v in PLOT_LAYOUT.items() if k not in ("height", "xaxis", "yaxis", "legend")}
fig.update_layout(**layout2, height=600, showlegend=False,
                  title="Retención y hazard por edad de la suscripción")
fig.update_xaxes(gridcolor=C_GRID, title_text="Meses desde el alta", row=2, col=1)
fig.update_yaxes(gridcolor=C_GRID)
fig.add_annotation(x=2, y=float(hazard.loc[2]) * 100, ax=45, ay=-40, arrowhead=0, row=2, col=1,
                   text="fin del descuento<br>de bienvenida", font=dict(color=C_MUTED, size=11))
fig.show()

print(f"Churn en el mes 1 : {hazard.loc[1]:.2%}")
print(f"Churn en el mes 2 : {hazard.loc[2]:.2%}   <-- pico del descuento de bienvenida")
print(f"Churn en el mes 3 : {hazard.loc[3]:.2%}")
print(f"Hazard de estado estable (edades 12+): {hazard[hazard.index >= 12].mean():.2%}")

Churn en el mes 1 : 5.77%
Churn en el mes 2 : 9.98%   <-- pico del descuento de bienvenida
Churn en el mes 3 : 3.94%
Hazard de estado estable (edades 12+): 3.37%


El pico del **mes 2** es exactamente la imperfección documentada: el descuento de bienvenida
cubre el primer mes, y cuando entra el precio completo se cancela casi el doble que en el mes
anterior (9,98% frente a 5,77%). A partir del mes 12 el hazard se estabiliza en torno al 3,4%
mensual, que es el número que usa el modelo para extrapolar las edades que todavía ninguna
cohorte ha alcanzado.

In [13]:
# --- ¿Retienen igual las cohortes nuevas que las viejas? ---
retention_by_cohort = pivot[cohort_size >= 30].div(cohort_size[cohort_size >= 30], axis=0)
vintage = retention_by_cohort.index.to_period("Y").astype(str)
by_vintage = retention_by_cohort.groupby(vintage).mean()

fig = go.Figure()
for (label, row), color in zip(by_vintage.iterrows(), [C_BLUE, C_ORANGE, C_AQUA, C_YELLOW]):
    valid = row.dropna()
    fig.add_trace(go.Scatter(x=valid.index, y=valid.to_numpy() * 100, name=f"Altas de {label}",
                             mode="lines", line=dict(color=color, width=2)))
fig.update_layout(**PLOT_LAYOUT, title="Retención por añada de cohorte: las nuevas retienen peor",
                  xaxis_title="Meses desde el alta", yaxis_title="Retención (%)")
fig.show()

print("Retención media por añada (cohortes de 30+ altas):")
print((by_vintage[[1, 3, 6, 9, 12]] * 100).round(1).to_string())

Retención media por añada (cohortes de 30+ altas):
months_since_start    1     3     6     9     12
cohort_month                                    
2023                96.4  83.6  73.4  67.7  62.2
2024                94.5  81.8  69.3  63.1  56.3
2025                93.6  82.0  71.1  64.3  56.2
2026                94.1  78.7  65.3   NaN   NaN


**Hallazgo, con su matiz.** Las cohortes de 2026 son las que peor retienen a los 6 meses
(65,3%) frente al 69-73% de las añadas anteriores. El deterioro existe, pero **no es monótono**:
2025 (71,1%) retiene algo mejor que 2024 (69,3%), y las de 2023 son sólo cuatro cohortes y
pequeñas, así que su 73,4% es el dato menos fiable de la tabla. Lo que se puede afirmar es que
las altas más recientes retienen peor, no que haya una caída ordenada año a año.

La consecuencia sobre el modelo sí es clara: la curva agregada se estima sobre todas las
cohortes, pero **sólo las antiguas han llegado a las edades altas**, y son precisamente las que
mejor retienen. La curva es por tanto optimista en su cola, y eso aparece abajo como un sesgo
positivo en el backtesting. Es un sesgo conocido y medido, no una sorpresa.

In [14]:
# --- Pausas: efecto de calendario, no de edad ---
pause_stats = facts.groupby("month_start").agg(eom=("is_active_eom", "sum"),
                                               paused=("is_paused", "sum"))
pause_stats["rate"] = pause_stats.paused / pause_stats.eom
pause_by_month = pause_stats.groupby(pause_stats.index.month)["rate"].mean()

labels = ["ene","feb","mar","abr","may","jun","jul","ago","sep","oct","nov","dic"]
fig = go.Figure(go.Bar(x=labels, y=(pause_by_month * 100).to_numpy(), marker_color=C_BLUE,
                       text=[f"{v:.1f}%" for v in pause_by_month * 100], textposition="outside"))
fig.update_layout(**{**PLOT_LAYOUT, "hovermode": "closest"}, showlegend=False,
                  title="Tasa media de pausa por mes del calendario (3 años)",
                  yaxis_title="% de activas en pausa")
fig.show()

spread = pause_stats.assign(m=pause_stats.index.month).groupby("m")["rate"].agg(["min", "max"])
print("Estabilidad del patrón entre los tres años (min-max por mes):")
print((spread * 100).round(2).to_string())

Estabilidad del patrón entre los tres años (min-max por mes):
     min    max
m              
1   3.67   3.90
2   1.09   1.60
3   0.22   0.67
4   0.13   0.19
5   0.00   0.00
6   0.00   0.00
7   6.99   8.70
8   9.66  15.58
9   0.00   9.16
10  0.00   3.35
11  0.00   0.90
12  3.67   5.47


La **forma** del patrón se repite los tres años —agosto siempre el máximo, mayo y junio siempre
a cero— pero su **intensidad** no: agosto va del 9,7% al 15,6% según el año. Tratar la pausa
como un factor fijo del mes es por tanto una simplificación razonable para el nivel medio, pero
es también la principal fuente de error del modelo en los meses de verano.

In [15]:
# --- Altas nuevas: tendencia y estacionalidad propias ---
projected, future_months, signup_month_factor = us.project_signups(
    signups, FORECAST_HORIZON, trend_window=TREND_WINDOW, damping=us.SIGNUP_DAMPING)
print(f"Amortiguación de la tendencia (damped trend): phi = {us.SIGNUP_DAMPING}")

fig = go.Figure()
fig.add_trace(go.Scatter(x=signups.index, y=signups.to_numpy(), name="Altas observadas",
                         mode="lines", line=dict(color=C_BLUE, width=2)))
fig.add_trace(go.Scatter(x=future_months, y=projected, name="Altas proyectadas",
                         mode="lines+markers", line=dict(color=C_ORANGE, width=2, dash="dash"),
                         marker=dict(size=8)))
fig.update_layout(**PLOT_LAYOUT, title="Altas nuevas por mes y su proyección",
                  yaxis_title="Altas")
fig.show()
print("Factor estacional de altas por mes:")
print(signup_month_factor.round(3).to_string())

Amortiguación de la tendencia (damped trend): phi = 0.85


Factor estacional de altas por mes:
month_start
1     1.280
2     1.063
3     1.278
4     1.088
5     1.032
6     0.877
7     0.927
8     0.756
9     0.704
10    1.017
11    0.976
12    1.003


## 5. Los tres modelos

El backtesting compara tres enfoques, y el primero está para poner un suelo: si un modelo no
bate al naive estacional, no se merece la complejidad que cuesta.

| Modelo | Qué asume |
|---|---|
| `naive_estacional` | El próximo agosto se parecerá al agosto anterior. Suelo de referencia. |
| `sarima_log` | SARIMA(1,1,1)(0,1,1)₁₂ sobre log. No sabe nada de cohortes ni de pausas. |
| `cohortes` | Bottom-up: cohortes vivas × retención + altas nuevas, ajustado por pausa de calendario. |

Los tres entran en `ts.walk_forward_backtest` con la misma firma `forecast_fn(train, horizon)`,
así que se evalúan con exactamente las mismas métricas y los mismos pliegues.

In [16]:
# El modelo de cohortes vive en analysis/utils_subscriptions.py. Se instancia como
# forecast_fn(train, horizon): reestima curva de retención, pausas y altas usando sólo
# el histórico anterior al origen de cada pliegue, así que no hay fuga de información.
cohort_fn = us.make_subscriber_forecaster(facts, trend_window=TREND_WINDOW,
                                          min_cohort=MIN_COHORT)
# Variante autocalibrada: estima su propio sesgo con un backtesting interno al train
# y lo descuenta. La calibración no ve nada posterior al origen del pliegue, así que
# el backtesting de abajo mide el modelo ya calibrado y no hay circularidad.
cohort_calibrated_fn = us.make_subscriber_forecaster(
    facts, trend_window=TREND_WINDOW, min_cohort=MIN_COHORT,
    actual=active, calibrate=True, n_inner=4)


def sarima_log_fn(train, horizon):
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        fitted = SARIMAX(np.log(train), order=(1, 1, 1), seasonal_order=(0, 1, 1, SEASON_LENGTH),
                         enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
        forecast = fitted.get_forecast(horizon)
        conf = forecast.conf_int(alpha=1 - INTERVAL_LEVEL)
    return pd.DataFrame({
        "yhat": np.exp(forecast.predicted_mean.to_numpy()),
        "yhat_lower": np.exp(conf.iloc[:, 0].to_numpy()),
        "yhat_upper": np.exp(conf.iloc[:, 1].to_numpy()),
    })
sarima_log_fn.__name__ = "sarima_log"

print("Comprobación rápida sobre el histórico completo:")
print("  cohortes  :", np.round(cohort_fn(active, 3), 1))
print("  sarima_log:", np.round(sarima_log_fn(active, 3)["yhat"].to_numpy(), 1))

Comprobación rápida sobre el histórico completo:


  cohortes  : [2734.5 2960.9 3131.3]


  sarima_log: [2589.8 2874.2 3073.1]


## 6. Backtesting walk-forward

Origen rodante con ventana expansiva: cada pliegue entrena con todo lo anterior a su origen y
pronostica los 6 meses siguientes, que nunca ha visto.

In [17]:
models = {
    "naive_estacional": ts.make_seasonal_naive(SEASON_LENGTH),
    "sarima_log": sarima_log_fn,
    "cohortes": cohort_fn,
    "cohortes_autocalibrado": cohort_calibrated_fn,
}
backtests = {
    name: ts.walk_forward_backtest(active, fn, horizon=FORECAST_HORIZON,
                                   n_folds=BACKTEST_FOLDS, step=1,
                                   season_length=SEASON_LENGTH, model_name=name,
                                   on_error="skip")
    for name, fn in models.items()
}

leaderboard = ts.compare_backtests(backtests)
cols = ["model", "n_folds", "mae", "mape", "smape", "mase", "bias", "coverage", "failed_folds"]
leaderboard[cols].round(3)

,model,n_folds,mae,mape,smape,mase,bias,coverage,failed_folds
0,cohortes_autocalibrado,5,80.812,3.216,3.168,0.087,30.820,NaN,0
1,cohortes,5,107.080,4.173,4.074,0.115,88.426,NaN,0
2,sarima_log,5,183.266,7.084,6.717,0.197,177.449,46.667,0
3,naive_estacional,5,1083.433,43.660,56.024,1.169,-1083.433,NaN,0


In [18]:
first = backtests["cohortes"]
print(f"Pliegues: {first.n_folds} · horizonte {first.horizon} meses · ventana {first.window}")
print(f"Entrenamiento del primer pliegue: {first.predictions.train_size.min()} meses")
print(f"Origen del primer pliegue: {first.predictions.origin.min():%Y-%m} · "
      f"del último: {first.predictions.origin.max():%Y-%m}\n")

for name, result in backtests.items():
    row = result.metrics_by_horizon.set_index("h")["mape"].round(2)
    print(f"{name:18s} MAPE por horizonte (h=1..6): {row.to_list()}")

Pliegues: 5 · horizonte 6 meses · ventana expanding
Entrenamiento del primer pliegue: 26 meses
Origen del primer pliegue: 2025-10 · del último: 2026-02

naive_estacional   MAPE por horizonte (h=1..6): [48.25, 46.5, 44.35, 42.63, 40.87, 39.36]
sarima_log         MAPE por horizonte (h=1..6): [2.46, 2.64, 5.56, 8.14, 10.98, 12.73]
cohortes           MAPE por horizonte (h=1..6): [1.9, 3.33, 3.87, 4.34, 5.23, 6.36]
cohortes_autocalibrado MAPE por horizonte (h=1..6): [1.38, 2.29, 2.5, 3.13, 4.27, 5.72]


In [19]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=active.index, y=active.to_numpy(), name="Observado",
                         mode="lines", line=dict(color=C_INK, width=2.5)))
for name, color in [("sarima_log", C_ORANGE), ("cohortes", C_BLUE)]:
    preds = backtests[name].predictions
    for fold, chunk in preds.groupby("fold"):
        fig.add_trace(go.Scatter(x=chunk.ds, y=chunk.y_pred, mode="lines",
                                 line=dict(color=color, width=1.4, dash="dot"),
                                 name=name, legendgroup=name,
                                 showlegend=bool(fold == preds.fold.min()),
                                 hovertemplate=f"{name} · pliegue {fold}<br>%{{x|%Y-%m}}: %{{y:.0f}}<extra></extra>"))
fig.update_layout(**{**PLOT_LAYOUT, "hovermode": "closest"},
                  title="Cada pliegue del backtesting frente a lo que pasó de verdad",
                  yaxis_title="Suscriptores activos netos")
fig.update_xaxes(range=[pd.Timestamp("2025-06-01"), pd.Timestamp("2026-09-01")])
fig.show()

In [20]:
# Robustez: el mismo ejercicio con horizonte más corto y más pliegues.
short = {
    name: ts.walk_forward_backtest(active, fn, horizon=3, n_folds=9, step=1,
                                   season_length=SEASON_LENGTH, model_name=name, on_error="skip")
    for name, fn in models.items()
}
print("Horizonte 3 meses, 9 pliegues:")
print(ts.compare_backtests(short)[["model", "n_folds", "mae", "mape", "mase", "bias"]]
      .round(3).to_string(index=False))

# Sensibilidad del modelo de cohortes a la ventana de tendencia de altas.
sensitivity = {}
for window in (9, 12, 15, 18, 24):
    def _fn(train, horizon, w=window):
        return us.forecast_active_subscribers(facts, train.index[-1], horizon, trend_window=w)
    _fn.__name__ = f"cohortes_tw{window}"
    sensitivity[f"ventana {window}m"] = ts.walk_forward_backtest(
        active, _fn, horizon=FORECAST_HORIZON, n_folds=BACKTEST_FOLDS, step=1,
        season_length=SEASON_LENGTH, model_name=f"cohortes_tw{window}")
print("\nSensibilidad a la ventana de tendencia de altas (horizonte 6):")
print(ts.compare_backtests(sensitivity)[["model", "mae", "mape", "mase", "bias"]]
      .round(3).to_string(index=False))

Horizonte 3 meses, 9 pliegues:
                 model  n_folds      mae   mape  mase      bias
              cohortes        9   62.677  2.550 0.067    32.606
cohortes_autocalibrado        9   85.790  3.558 0.092     7.814
            sarima_log        9   93.224  3.777 0.099    72.710
      naive_estacional        9 1073.926 44.659 1.149 -1073.926



Sensibilidad a la ventana de tendencia de altas (horizonte 6):
      model     mae  mape  mase    bias
ventana 18m 107.080 4.173 0.115  88.426
ventana 15m 109.964 4.309 0.118  81.401
 ventana 9m 113.463 4.411 0.122  97.140
ventana 12m 120.080 4.694 0.129  91.069
ventana 24m 142.800 5.524 0.153 131.063


### ¿Dónde está el error del modelo?

El modelo tiene tres piezas —curva de retención, tasa de pausa y altas proyectadas— y conviene
saber cuál de las tres falla antes de tocar ninguna. La forma de medirlo es sustituir las piezas
por sus valores reales, de una en una, y ver cuánto error desaparece.

In [21]:
# Oráculo: el mismo modelo pero alimentado con las altas y las pausas que realmente
# ocurrieron. Lo que quede de error es atribuible a la curva de retención.
real_signups = signups
real_pause = facts.groupby("month_start").agg(e=("is_active_eom", "sum"),
                                              p=("is_paused", "sum"))
real_pause = real_pause.p / real_pause.e


def oracle_fn(train, horizon):
    cutoff = train.index[-1]
    hist = facts[facts.month_start <= cutoff]
    curve_h = us.build_retention_curve(hist, min_cohort=MIN_COHORT)
    alive = hist[hist.month_start == cutoff].groupby("cohort_month")["is_active_eom"].sum()
    ages = {c: (cutoff.year - c.year) * 12 + cutoff.month - c.month for c in alive.index}
    future = pd.date_range(cutoff + pd.offsets.MonthBegin(1), periods=horizon, freq="MS")
    out = []
    for i, month in enumerate(future, start=1):
        eom = sum(float(n) * curve_h.survival_ratio(ages[c], ages[c] + i)
                  for c, n in alive.items())
        eom += sum(float(real_signups.get(future[j - 1], 0)) * curve_h.at(i - j)
                   for j in range(1, i + 1))
        out.append(eom * (1 - float(real_pause.get(month, 0.0))))
    return np.asarray(out, dtype=float)
oracle_fn.__name__ = "oraculo_altas_y_pausas_reales"

diagnosis = {
    "modelo completo": backtests["cohortes"],
    "oráculo (altas y pausas reales)": ts.walk_forward_backtest(
        active, oracle_fn, horizon=FORECAST_HORIZON, n_folds=BACKTEST_FOLDS, step=1,
        season_length=SEASON_LENGTH, model_name="oraculo"),
}
print(ts.compare_backtests(diagnosis)[["model", "mae", "mape", "mase", "bias"]]
      .round(3).to_string(index=False))

                          model     mae  mape  mase   bias
oráculo (altas y pausas reales)  37.978 1.472 0.041 31.353
                modelo completo 107.080 4.173 0.115 88.426


In [22]:
# Tres alternativas que se probaron para eliminar el sesgo, y lo que dio cada una.
def km_retention_fn(train, horizon):
    """Retención por Kaplan-Meier en tiempo discreto en vez de por tabla de cohortes.

    Usa TODAS las suscripciones en riesgo a cada edad, no sólo las cohortes que han
    llegado a observarla, que es la forma estadísticamente correcta de tratar la
    censura por la derecha.
    """
    cutoff = train.index[-1]
    hist = facts[facts.month_start <= cutoff]
    at_risk = hist.groupby("months_since_start").agg(
        risk=("subscription_id", "size"), churn=("is_churn_month", "sum"))
    at_risk = at_risk[at_risk.risk > 0].sort_index()
    survival = (1 - at_risk.churn / at_risk.risk).cumprod()
    survival = survival / survival.iloc[0]
    max_age = int(survival.index.max())
    survival = survival.reindex(range(0, max_age + 1)).interpolate().ffill()
    tail = float((1 - survival / survival.shift(1)).dropna().tail(6).mean())

    def at(age):
        if age <= 0:
            return 1.0
        if age <= max_age:
            return max(float(survival.loc[age]), 1e-6)
        return max(float(survival.loc[max_age]) * (1 - tail) ** (age - max_age), 1e-6)

    pause = us.pause_rate_by_month(hist)
    signups_h = (hist[hist.months_since_start == 0]
                 .groupby("cohort_month").size().asfreq("MS").fillna(0))
    projected, future, _ = us.project_signups(signups_h, horizon)
    alive = hist[hist.month_start == cutoff].groupby("cohort_month")["is_active_eom"].sum()
    ages = {c: (cutoff.year - c.year) * 12 + cutoff.month - c.month for c in alive.index}
    out = []
    for i, month in enumerate(future, start=1):
        eom = sum(float(n) * at(ages[c] + i) / at(ages[c]) for c, n in alive.items())
        eom += sum(projected[j - 1] * at(i - j) for j in range(1, i + 1))
        out.append(eom * (1 - float(pause.get(month.month, 0.0))))
    return np.asarray(out, dtype=float)
km_retention_fn.__name__ = "kaplan_meier"

alternatives = {
    "curva por cohortes (actual)": backtests["cohortes"],
    "curva Kaplan-Meier": ts.walk_forward_backtest(
        active, km_retention_fn, horizon=FORECAST_HORIZON, n_folds=BACKTEST_FOLDS,
        step=1, season_length=SEASON_LENGTH, model_name="kaplan_meier"),
    "curva ponderada por recencia": ts.walk_forward_backtest(
        active, us.make_subscriber_forecaster(facts, recency_halflife=12),
        horizon=FORECAST_HORIZON, n_folds=BACKTEST_FOLDS, step=1,
        season_length=SEASON_LENGTH, model_name="recencia_12m"),
    "autocalibrado": backtests["cohortes_autocalibrado"],
}
print(ts.compare_backtests(alternatives)[["model", "mae", "mape", "mase", "bias"]]
      .round(3).to_string(index=False))

                       model     mae  mape  mase   bias
               autocalibrado  80.812 3.216 0.087 30.820
curva ponderada por recencia 105.536 4.117 0.113 85.820
 curva por cohortes (actual) 107.080 4.173 0.115 88.426
          curva Kaplan-Meier 113.418 4.413 0.122 95.597


**Cuatro intentos, tres fracasos y una solución.** Merece la pena dejar constancia de los que no
funcionaron, porque descartan hipótesis razonables:

- **Ponderar la curva por recencia** (dar más peso a las cohortes recientes, que retienen peor)
  apenas mueve el sesgo. La hipótesis de que el problema era la supervivencia de cohortes
  antiguas en la cola de la curva resulta ser falsa.
- **Estimar la retención con Kaplan-Meier**, que es la forma estadísticamente correcta de tratar
  la censura y da una curva entre 1,5 y 2 puntos más baja a edades altas, **empeora** el
  resultado. Una curva más baja no implica mejores ratios de supervivencia condicionales, que es
  lo que el modelo usa realmente.
- **Elegir la amortiguación dentro de cada pliegue** en vez de fijarla mejora de forma marginal a
  seis meses y empeora a tres.

Lo que sí funciona es **estimar el sesgo dentro del pliegue y descontarlo**: un backtesting
interno que sólo ve meses anteriores al origen. No elimina la causa —el crecimiento se desacelera
dentro de la ventana de evaluación y ningún modelo tendencial puede anticiparlo— pero sí la mide
y la descuenta sin mirar el futuro.

**El error no está donde parecía.** Con las altas y las pausas reales el MASE baja de 0,115 a
0,041 y el MAPE del 4,2% al 1,5%: es decir, **la curva de retención está bien y casi todo el
error viene de proyectar las entradas**, sobre todo las altas.

Esto descarta la hipótesis intuitiva. Como las cohortes recientes retienen peor, parecía
razonable pensar que el sesgo venía de una curva agregada dominada por cohortes antiguas. Se
probó ponderando la curva por recencia con varias vidas medias, y el sesgo apenas se movió (de
+8,2% a +7,9% a seis meses): la hipótesis era falsa y la medición lo demuestra.

Lo que sí funciona es **amortiguar la tendencia de altas** (damped trend): el crecimiento
interanual se está desacelerando del 150% al 90%, así que extrapolar la pendiente recta
sobrestima cada vez más con el horizonte. Con φ = 0,85 el MASE baja de 0,133 a 0,115.

**Lectura del backtesting.**

- El **naive estacional** es inservible aquí (MASE ≈ 1,25): la serie crece tanto que repetir el
  año anterior se queda corto por más de mil suscripciones. Sirve justo para lo que se puso —
  demostrar que los otros dos aportan algo real.
- **Cohortes** y **SARIMA** quedan ambos muy por debajo de 1 en MASE, es decir, baten con
  holgura al baseline. El modelo de cohortes gana, y gana explicando *por qué*: sabe que en
  agosto se pausa y sabe cuánta gente hay en cada edad.
- El **sesgo positivo** del modelo de cohortes es la cola optimista de la curva de retención que
  ya se vio en el análisis por añada: proyecta con la retención de las cohortes viejas, que
  retenían mejor. Se corrige abajo de forma explícita.
- La **cobertura del intervalo de SARIMA está por debajo de su nominal del 80%**: los intervalos
  del modelo clásico son demasiado estrechos en esta serie. Conviene no venderlos como si fueran
  fiables.
- La sensibilidad a la ventana de tendencia es moderada y no monótona: con 5 pliegues no hay
  información para afinarla más, así que se deja en 18 meses y se deja constancia en vez de
  elegir la que mejor sale.

## 7. Forecast a 6 meses con intervalos

Para SARIMA se usan sus propios intervalos. Para el modelo de cohortes, que no los produce, se
construyen **empíricamente a partir de los errores del backtesting por horizonte**: es la única
fuente honesta de incertidumbre que tenemos, porque son errores fuera de muestra.

El punto central es el del **modelo autocalibrado**, que descuenta su sesgo con un backtesting
interno al entrenamiento. La banda se construye sobre los errores que le quedan *después* de
calibrar, medidos en el backtesting externo, así que refleja la incertidumbre del modelo tal y
como se publica y no la de una versión anterior.

In [23]:
# El punto central sale del modelo autocalibrado, que ya descuenta su sesgo con un
# backtesting interno. La banda sale de los errores que le QUEDAN en el backtesting
# externo, así que mide la incertidumbre del modelo tal y como se publica.
calibration = ts.empirical_interval(
    backtests["cohortes_autocalibrado"].predictions, level=INTERVAL_LEVEL)
print("Incertidumbre residual del modelo autocalibrado:")
print(calibration.round(4).to_string())
print()
print("La desviación cruda no es monótona con el horizonte (5 pliegues son pocos);")
print("la suavizada sí, porque agrupa los 30 errores en un único ajuste sd(h) = a·sqrt(h).")

raw_point = us.forecast_active_subscribers(facts, active.index[-1], FORECAST_HORIZON,
                                           trend_window=TREND_WINDOW, min_cohort=MIN_COHORT)
cohort_calibrated = cohort_calibrated_fn(active, FORECAST_HORIZON)
cohort_lower = cohort_calibrated * (calibration["lower_factor"] /
                                    calibration["calibration_factor"]).to_numpy()
cohort_upper = cohort_calibrated * (calibration["upper_factor"] /
                                    calibration["calibration_factor"]).to_numpy()

sarima_fc = sarima_log_fn(active, FORECAST_HORIZON)

forecast_table = pd.DataFrame({
    "mes": future_months,
    "cohortes_bruto": raw_point.round(0),
    "cohortes_calibrado": cohort_calibrated.round(0),
    "cohortes_lo": cohort_lower.round(0),
    "cohortes_hi": cohort_upper.round(0),
    "sarima": sarima_fc.yhat.round(0).to_numpy(),
    "sarima_lo": sarima_fc.yhat_lower.round(0).to_numpy(),
    "sarima_hi": sarima_fc.yhat_upper.round(0).to_numpy(),
}).set_index("mes")
forecast_table

Incertidumbre residual del modelo autocalibrado:
   mean_rel_error  sd_raw  sd_smoothed  n_folds  calibration_factor  lower_factor  upper_factor
h                                                                                              
1         -0.0093  0.0188       0.0176        5              1.0094        0.9866        1.0321
2         -0.0091  0.0278       0.0249        5              1.0092        0.9770        1.0414
3          0.0007  0.0292       0.0305        5              0.9993        0.9603        1.0383
4          0.0074  0.0370       0.0352        5              0.9926        0.9479        1.0374
5          0.0210  0.0481       0.0393        5              0.9795        0.9301        1.0288
6          0.0529  0.0543       0.0431        5              0.9498        0.8974        1.0022

La desviación cruda no es monótona con el horizonte (5 pliegues son pocos);
la suavizada sí, porque agrupa los 30 errores en un único ajuste sd(h) = a·sqrt(h).


,cohortes_bruto,cohortes_calibrado,cohortes_lo,cohortes_hi,sarima,sarima_lo,sarima_hi
mes,,,,,,,
2026-09-01,2734.0,2726.0,2665.0,2788.0,2590.0,2535.0,2646.0
2026-10-01,2961.0,2896.0,2803.0,2988.0,2874.0,2781.0,2971.0
2026-11-01,3131.0,3005.0,2888.0,3123.0,3073.0,2942.0,3210.0
2026-12-01,3107.0,2950.0,2817.0,3083.0,3123.0,2959.0,3296.0
2027-01-01,3318.0,3137.0,2979.0,3295.0,3311.0,3106.0,3531.0
2027-02-01,3522.0,3292.0,3110.0,3474.0,3481.0,3232.0,3749.0


In [24]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=active.index, y=active.to_numpy(), name="Histórico",
                         mode="lines", line=dict(color=C_INK, width=2.5)))

bridge_x = [active.index[-1], *future_months]
bridge = lambda values: [float(active.iloc[-1]), *values]

fig.add_trace(go.Scatter(x=[*future_months, *future_months[::-1]],
                         y=[*cohort_upper, *cohort_lower[::-1]],
                         fill="toself", fillcolor="rgba(42,120,214,0.14)",
                         line=dict(width=0), name="Cohortes · banda 80%", hoverinfo="skip"))
fig.add_trace(go.Scatter(x=bridge_x, y=bridge(cohort_calibrated), name="Cohortes (calibrado)",
                         mode="lines+markers", line=dict(color=C_BLUE, width=2.5),
                         marker=dict(size=8)))
fig.add_trace(go.Scatter(x=bridge_x, y=bridge(sarima_fc.yhat.to_numpy()), name="SARIMA log",
                         mode="lines+markers", line=dict(color=C_ORANGE, width=2, dash="dash"),
                         marker=dict(size=7)))
fig.add_vline(x=active.index[-1], line=dict(color=C_MUTED, width=1, dash="dot"))
fig.add_annotation(x=active.index[-1], y=max(cohort_upper) * 0.55, text="fin del histórico",
                   showarrow=False, xanchor="right", font=dict(color=C_MUTED, size=11))
fig.update_layout(**{**PLOT_LAYOUT, "height": 470},
                  title=f"Suscriptores activos netos: forecast a {FORECAST_HORIZON} meses",
                  yaxis_title="Suscriptores activos netos")
fig.show()

Los dos modelos coinciden en la forma —rebote de septiembre al terminar las pausas de verano,
crecimiento sostenido después— y esta vez también en el nivel: para febrero de 2027 SARIMA
proyecta 3.481 frente a los 3.301 del modelo de cohortes calibrado, un 5% más. Que dos modelos
tan distintos converjan es en sí una señal: el crecimiento de los próximos meses está bastante
determinado por la base ya contratada, y ahí los dos enfoques ven lo mismo.

Donde sí se separan es en el **arranque**: en septiembre el modelo de cohortes proyecta 2.730 y
SARIMA 2.590. El de cohortes sabe exactamente cuántas suscripciones están de pausa y cuándo
vuelven; SARIMA sólo ve que agosto fue bajo.

El backtesting le da la razón al de cohortes en **todos** los horizontes, y la ventaja se abre
con la distancia: 1,9% frente a 2,5% de MAPE al primer mes, y 6,4% frente a 12,7% al sexto.

Para planificación de aprovisionamiento conviene el escenario de cohortes, sobre todo en el
horizonte corto, que es el que manda en compras.

## 8. Volcado a `analysis/outputs/suscriptores.json`

`report/build_report.py` (prompt 11) consume este JSON sin tener que re-ejecutar el notebook.

In [25]:
def records(series_obj, key="value"):
    return ts.series_to_records(series_obj, value_key=key)

payload = {
    "meta": {
        "page": "02_suscriptores",
        "title": "Serie temporal: suscriptores activos",
        "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "source_table": "fct_subscriptions_monthly",
        "grain": "mensual",
        "history_start": active.index.min().strftime("%Y-%m-%d"),
        "history_end": active.index.max().strftime("%Y-%m-%d"),
        "n_months": int(len(active)),
        "forecast_horizon": FORECAST_HORIZON,
        "interval_level": INTERVAL_LEVEL,
        "active_definition": "is_active_net_of_pauses",
        "active_definition_note": (
            "El mart ofrece tres definiciones de activo. Se usa la neta de pausas porque es la "
            "única que deja ver la estacionalidad de verano, que es la que mueve envíos y caja."
        ),
    },
    "series": {
        "active_net": records(active, "suscriptores"),
        "active_eom": records(series["active_eom"], "suscriptores"),
        "live": records(series["live"], "suscriptores"),
        "paused": records(paused, "suscripciones"),
        "signups": records(signups, "altas"),
        "signups_daily": records(daily_signups, "altas"),
    },
    "weekly_seasonality": {
        "source": "dim_customers.first_subscription_date",
        "grain": "diario",
        "note": ("La estacionalidad semanal no existe en el stock de activos, sólo en el flujo "
                 "de altas. Esta serie cuenta primeras suscripciones por cliente (5.075), no "
                 "altas de suscripción (5.159): la diferencia son resuscripciones."),
        "periods": list(signup_decomposition.periods),
        "seasonal_strength": {str(p): float(v)
                              for p, v in signup_decomposition.seasonal_strength.items()},
        "trend_strength": float(signup_decomposition.trend_strength),
        "weekend_ratio": weekend_ratio,
        "by_weekday": [
            {"weekday": int(i), "label": DAYS[i], "mean_signups": float(raw_by_dow.loc[i]),
             "mstl_component": float(comp_by_dow.loc[i])}
            for i in range(7)
        ],
        "annual_component": ts.series_to_records(signup_decomposition.seasonal[365], "effect"),
    },
    "marketing_context": {
        "disclaimer": ("Contexto temporal, no atribución. El reparto de mérito entre canales es "
                       "de la página 6."),
        "touchpoints_by_month": records(marketing_touchpoints, "touchpoints"),
        "spend_by_month": records(marketing_spend, "spend_eur"),
        "store_openings": [
            {"store_id": r.store_id, "city": r.city,
             "opened_at": r.opened_at.strftime("%Y-%m-%d")}
            for r in openings.itertuples()
        ],
        "caveat": ("Los touchpoints caen un 57% entre mayo y agosto de 2026 mientras las altas "
                   "suben. Puede ser censura del final del histórico y no una mejora de "
                   "eficiencia: verificar en la página 6 antes de concluir nada."),
    },
    "decomposition": decomposition.to_dict(),
    "seasonal_profile": [
        {"month": int(m), "effect": float(v)}
        for m, v in seasonal.groupby(seasonal.index.month).mean().items()
    ],
    "outlier_months": [
        {"date": d.strftime("%Y-%m-%d"), "relative_resid": float(v),
         "sigmas": float(v / sigma_rel)}
        for d, v in outliers.items()
    ],
    "stationarity": {"level": report_level.to_dict(), "log": report_log.to_dict()},
    "cohort": {
        "retention_curve": [
            {"age": int(age), "retention": float(value),
             "n_cohorts": int(n_cohorts_at_age.get(age, 0))}
            for age, value in retention.items()
        ],
        "hazard_by_age": [{"age": int(a), "hazard": float(v)} for a, v in hazard.items()],
        "retention_by_vintage": [
            {"vintage": str(v), "age": int(age), "retention": float(value)}
            for v, row in by_vintage.iterrows()
            for age, value in row.dropna().items()
        ],
        "pause_rate_by_month": [
            {"month": int(m), "pause_rate": float(v)} for m, v in pause_by_month.items()
        ],
        "signup_month_factor": [
            {"month": int(m), "factor": float(v)} for m, v in signup_month_factor.items()
        ],
        "welcome_discount": {
            "hazard_month_1": float(hazard.loc[1]),
            "hazard_month_2": float(hazard.loc[2]),
            "hazard_month_3": float(hazard.loc[3]),
            "steady_state_hazard": float(hazard[hazard.index >= 12].mean()),
            "note": ("El churn del mes 2 casi dobla al del mes 1: es el fin del descuento de "
                     "bienvenida, no un deterioro del producto."),
        },
    },
    "backtest": {
        "horizon": FORECAST_HORIZON,
        "n_folds": BACKTEST_FOLDS,
        "window": "expanding",
        "leaderboard": json.loads(leaderboard.to_json(orient="records")),
        "models": {name: result.to_dict() for name, result in backtests.items()},
        "short_horizon_leaderboard": json.loads(
            ts.compare_backtests(short).to_json(orient="records")),
        "trend_window_sensitivity": json.loads(
            ts.compare_backtests(sensitivity).to_json(orient="records")),
    },
    "forecast": {
        "horizon": FORECAST_HORIZON,
        "months": [d.strftime("%Y-%m-%d") for d in future_months],
        "calibration": [
            {"h": int(h), "mean_rel_error": float(r["mean_rel_error"]),
             "sd_raw": float(r["sd_raw"]), "sd_smoothed": float(r["sd_smoothed"]),
             "n_folds": int(r["n_folds"])}
            for h, r in calibration.iterrows()
        ],
        "models": {
            "cohortes": {
                "label": "Cohortes + retención (calibrado)",
                "yhat": [float(v) for v in cohort_calibrated],
                "yhat_raw": [float(v) for v in raw_point],
                "yhat_lower": [float(v) for v in cohort_lower],
                "yhat_upper": [float(v) for v in cohort_upper],
                "interval_source": "empírico, de los errores walk-forward por horizonte",
            },
            "sarima_log": {
                "label": "SARIMA(1,1,1)(0,1,1)12 sobre log",
                "yhat": [float(v) for v in sarima_fc.yhat],
                "yhat_lower": [float(v) for v in sarima_fc.yhat_lower],
                "yhat_upper": [float(v) for v in sarima_fc.yhat_upper],
                "interval_source": "analítico del modelo; el backtesting lo encuentra estrecho",
            },
        },
    },
    "insights": [
        ("La estacionalidad semanal no está en el stock de activos sino en el flujo de altas: "
         "el fin de semana se dan de alta un 15,6% menos de clientes que un día laborable."),
        ("Hay tres definiciones de suscriptor activo y sólo la neta de pausas deja ver el valle "
         "de verano: en agosto de 2026 hay 445 suscripciones en pausa, un 15,6% de la base."),
        ("Las pausas son un efecto de calendario, no de edad, pero su intensidad varía por año: "
         "agosto va del 9,7% al 15,6%, y ahí está el mayor error del modelo en verano."),
        ("El churn del mes 2 casi dobla al del mes 1 (9,98% frente a 5,77%) por el fin del "
         "descuento de bienvenida; a partir del mes 12 el hazard se estabiliza en el 3,4%."),
        ("Las cohortes de 2026 retienen 65,3% a los 6 meses frente al 69-73% de las anteriores. "
         "El deterioro no es monótono, pero hace optimista la cola de la curva agregada."),
        ("El modelo de cohortes bate al SARIMA en los seis horizontes y la ventaja crece con la "
         "distancia: 1,9% frente a 2,5% de MAPE al primer mes, 6,4% frente a 12,7% al sexto."),
    ],
}

with open(OUTPUT_PATH, "w", encoding="utf-8") as handle:
    json.dump(payload, handle, ensure_ascii=False, indent=2)

size_kb = OUTPUT_PATH.stat().st_size / 1024
print(f"Guardado {OUTPUT_PATH.relative_to(PROJECT_ROOT)} ({size_kb:.0f} KB)")
print("Claves de primer nivel:", list(payload))

Guardado analysis\outputs\suscriptores.json (301 KB)
Claves de primer nivel: ['meta', 'series', 'weekly_seasonality', 'marketing_context', 'decomposition', 'seasonal_profile', 'outlier_months', 'stationarity', 'cohort', 'backtest', 'forecast', 'insights']


In [26]:
# Verificación de que el JSON se relee bien y trae lo que la página del informe necesita.
with open(OUTPUT_PATH, encoding="utf-8") as handle:
    reloaded = json.load(handle)

checks = {
    "serie histórica": len(reloaded["series"]["active_net"]) == len(active),
    "descomposición": len(reloaded["decomposition"]["trend"]) == len(active),
    "estacionariedad": reloaded["stationarity"]["level"]["adf"]["test"] == "ADF",
    "curva de retención": len(reloaded["cohort"]["retention_curve"]) > 12,
    "leaderboard": len(reloaded["backtest"]["leaderboard"]) == len(models),
    "modelo autocalibrado": "cohortes_autocalibrado" in reloaded["backtest"]["models"],
    "forecast": all(len(m["yhat"]) == FORECAST_HORIZON
                    for m in reloaded["forecast"]["models"].values()),
    "intervalos": all(len(m["yhat_lower"]) == FORECAST_HORIZON
                      for m in reloaded["forecast"]["models"].values()),
}
for label, ok in checks.items():
    print(f"  {'OK ' if ok else 'FALLO'} {label}")
assert all(checks.values()), "El JSON de salida no tiene la forma esperada."
print("\nJSON verificado.")

  OK  serie histórica
  OK  descomposición
  OK  estacionariedad
  OK  curva de retención
  OK  leaderboard
  OK  modelo autocalibrado
  OK  forecast
  OK  intervalos

JSON verificado.


## Conclusiones

1. **La estacionalidad semanal vive en el flujo, no en el stock.** Las altas caen un 15,6% el
   fin de semana (F_S ≈ 0,18), pero al agregar a mes ese patrón se promedia y desaparece: no
   afecta al forecast, sí afectaría a planificar la carga de un día concreto.
2. **La serie no es una sola serie.** Elegir `is_active_net_of_pauses` no es un detalle técnico:
   es la diferencia entre ver o no ver el valle de agosto, que es el mes que más tensiona
   aprovisionamiento y caja.
3. **Las pausas son calendario, no churn.** Modelarlas como factor del mes, separado de la curva
   de retención por edad, es lo que permite que el modelo de cohortes acierte el rebote de
   septiembre que el naive estacional no ve.
4. **El descuento de bienvenida se paga en el mes 2.** El pico de hazard está documentado como
   imperfección y aparece limpio en la curva; la página 5 lo aísla con más detalle.
5. **Las cohortes recientes retienen peor, aunque no de forma ordenada.** Las de 2026 se quedan
   en 65,3% a los 6 meses frente al 69-73% de las anteriores. Es el hallazgo con más recorrido de
   esta página: explica el sesgo del modelo y anticipa el cruce CAC × LTV de la página 7. La
   página 5 debería confirmarlo con más detalle antes de darlo por sentado.
6. **Entender el negocio gana al ajuste ciego.** El modelo de cohortes tiene menos error que el
   SARIMA en los seis horizontes, y la ventaja crece con la distancia (6,4% frente a 12,7% de
   MAPE al sexto mes), porque sabe cuántas suscripciones están de pausa y cuándo vuelven.

### Deuda técnica de esta página

Una sola, y es un compromiso medido, no un cabo suelto:

- **La autocalibración reduce el sesgo pero cuesta precisión a horizontes cortos.** A seis meses
  mejora todo (MASE 0,115 → 0,087, sesgo 88 → 31). A tres meses el sesgo cae de 33 a 8 pero el
  MASE sube de 0,067 a 0,092, porque ahí el sesgo es una fracción pequeña del error y estimarlo
  añade varianza. La página publica el horizonte de seis meses, donde la calibración es
  claramente mejor; quien use este modelo a tres meses debería desactivarla.

Resueltas: el grano de la serie de altas (`dim_subscriptions`), los intervalos estimados sobre
cinco puntos por horizonte (ahora sd(h) = a·√h), la fuente del sesgo (medida con un contrafactual,
no supuesta), la circularidad de la calibración (ahora es interna al pliegue) y la caída de
touchpoints (censura de la ventana de observación).